In [ ]:
import pandas as pd

In [ ]:
df = pd.read_excel('/content/pos_tagged.xlsx')
display(df.head())

,Value
0,1\QT_QTC .\RD_PUNC 0\QT_QTC .\RD_PUNC
1,थेवबो\CC_CCS 1658\N_NN मायथायाव\N_NN जन\N_NNP ...
2,मुलुगनि\N_NN गुबुन\JJ गुबुन\V_VAUX थुनलाइनि\N_...
3,खुगाजों\N_NN खुगा\N_NN सोलिबोनाय\V_VM गथ’\N_NN...
4,मिसनारिफोरा\N_NN रनसायनाय\V_VM आरो\CC_CCD रावस...


In [ ]:
def extract_words_and_tokens(text):
    if pd.isna(text):
        return [], []

    words = []
    tokens = []

    for pair in text.split():

        if "\\" not in pair:
            continue

        word, tag = pair.rsplit("\\", 1)   # split only on the last backslash

        words.append(word)
        tokens.append(tag)

    return words, tokens

# Apply the function to the 'Value' column
df[['Words', 'Tokens']] = df['Value'].apply(lambda x: pd.Series(extract_words_and_tokens(x)))

display(df.head())

,Value,Words,Tokens
0,1\QT_QTC .\RD_PUNC 0\QT_QTC .\RD_PUNC,"[1, ., 0, .]","[QT_QTC, RD_PUNC, QT_QTC, RD_PUNC]"
1,थेवबो\CC_CCS 1658\N_NN मायथायाव\N_NN जन\N_NNP ...,"[थेवबो, 1658, मायथायाव, जन, असम, कमेनस्कि, (, ...","[CC_CCS, N_NN, N_NN, N_NNP, N_NNP, N_NNP, RD_P..."
2,मुलुगनि\N_NN गुबुन\JJ गुबुन\V_VAUX थुनलाइनि\N_...,"[मुलुगनि, गुबुन, गुबुन, थुनलाइनि, बादिनो, बर’,...","[N_NN, JJ, V_VAUX, N_NNP, PSP, N_NNP, N_NNP, N..."
3,खुगाजों\N_NN खुगा\N_NN सोलिबोनाय\V_VM गथ’\N_NN...,"[खुगाजों, खुगा, सोलिबोनाय, गथ’, थुनलाया, सिगां...","[N_NN, N_NN, V_VM, N_NNP, N_NNP, N_NST, V_VM, ..."
4,मिसनारिफोरा\N_NN रनसायनाय\V_VM आरो\CC_CCD रावस...,"[मिसनारिफोरा, रनसायनाय, आरो, रावसोलायनाय, बाइब...","[N_NN, V_VM, CC_CCD, V_VM, N_NNP, N_NN, RD_PUN..."


In [ ]:
print(df['Value'].iloc[1])

थेवबो\CC_CCS 1658\N_NN मायथायाव\N_NN जन\N_NNP असम\N_NNP कमेनस्कि\N_NNP (\RD_PUNC Jan\N_NNP Amos\N_NNP Komensky\N_NNP )\RD_PUNC आ\PSP फोसावनाय\V_VM सावगारि\N_NN बिजाब\N_NN Orbis\N_NNP Pictus\N_NNP खौनो\PSP गथ’\N_NNP थुनलाइनि\N_NNP गिबि\JJ बिजाब\N_NN होन्ना\V_VM साननाय\V_VM जायो\V_VM ।\RD_PUNC


In [ ]:
df[['Words','Tokens']] = df['Value'].apply(
    lambda x: pd.Series(extract_words_and_tokens(x))
)

In [ ]:
print(df.loc[1, "Words"])
print(df.loc[1, "Tokens"])

['थेवबो', '1658', 'मायथायाव', 'जन', 'असम', 'कमेनस्कि', '(', 'Jan', 'Amos', 'Komensky', ')', 'आ', 'फोसावनाय', 'सावगारि', 'बिजाब', 'Orbis', 'Pictus', 'खौनो', 'गथ’', 'थुनलाइनि', 'गिबि', 'बिजाब', 'होन्ना', 'साननाय', 'जायो', '।']
['CC_CCS', 'N_NN', 'N_NN', 'N_NNP', 'N_NNP', 'N_NNP', 'RD_PUNC', 'N_NNP', 'N_NNP', 'N_NNP', 'RD_PUNC', 'PSP', 'V_VM', 'N_NN', 'N_NN', 'N_NNP', 'N_NNP', 'PSP', 'N_NNP', 'N_NNP', 'JJ', 'N_NN', 'V_VM', 'V_VM', 'V_VM', 'RD_PUNC']


In [ ]:
from sklearn.model_selection import train_test_split

# Convert DataFrame columns to Python lists
sentences = df["Words"].tolist()
tags = df["Tokens"].tolist()

# First split: 80% train, 20% temp
train_sentences, temp_sentences, train_tags, temp_tags = train_test_split(
    sentences,
    tags,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

# Second split: temp into validation and test (10% each)
val_sentences, test_sentences, val_tags, test_tags = train_test_split(
    temp_sentences,
    temp_tags,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print(f"Training sentences   : {len(train_sentences)}")
print(f"Validation sentences : {len(val_sentences)}")
print(f"Testing sentences    : {len(test_sentences)}")

Training sentences   : 4804
Validation sentences : 600
Testing sentences    : 601


In [ ]:
word2idx = {
    "<PAD>": 0,
    "<UNK>": 1
}

for sentence in train_sentences:
    for word in sentence:
        if word not in word2idx:
            word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}

In [ ]:
tag2idx = {
    "<PAD>": 0
}

for sentence in train_tags:
    for tag in sentence:
        if tag not in tag2idx:
            tag2idx[tag] = len(tag2idx)

idx2tag = {idx: tag for tag, idx in tag2idx.items()}

In [ ]:
print("Vocabulary size :", len(word2idx))
print("Number of POS tags :", len(tag2idx))

print("\nSample word mapping:")
for i, (word, idx) in enumerate(word2idx.items()):
    print(f"{word} -> {idx}")
    if i == 10:
        break

print("\nTag mapping:")
print(tag2idx)

Vocabulary size : 18490
Number of POS tags : 36

Sample word mapping:
<PAD> -> 0
<UNK> -> 1
बे -> 2
बाथ्राखौ -> 3
मोनसे -> 4
गोजौ -> 5
थाखोनि -> 6
बिजिरनाय -> 7
एबा -> 8
स्टादिनिफ्राय -> 9
मिथिनो -> 10

Tag mapping:
{'<PAD>': 0, 'DM_DMD': 1, 'N_NST': 2, 'QT_QTC': 3, 'JJ': 4, 'V_VM': 5, 'CC_CCD': 6, 'RD_UNK': 7, 'RD_PUNC': 8, 'N_NN': 9, 'N_NNP': 10, 'RB': 11, 'PR_PRP': 12, 'PSP': 13, 'QT_QTO': 14, 'V_VAUX': 15, 'RD_ECH': 16, 'QT_QTF': 17, 'PR_PRI': 18, 'RD_RDF': 19, 'PR_PRF': 20, 'RD_SYM': 21, 'PR_PRC': 22, 'DM_DMQ': 23, 'PR_PRQ': 24, 'RP_RPD': 25, 'CC_CCS': 26, 'RP_NEG': 27, 'RP_INTF': 28, 'DM_DMI': 29, 'PR_PRL': 30, 'DM_DMR': 31, 'V_VM_VNF': 32, 'RP_INJ': 33, 'V_VM_VF': 34, 'V_VAUX_VF': 35}


In [ ]:
def encode_sentences(sentences, word2idx):
    encoded = []

    for sentence in sentences:
        encoded.append([
            word2idx.get(word, word2idx["<UNK>"])
            for word in sentence
        ])

    return encoded

In [ ]:
train_encoded = encode_sentences(train_sentences, word2idx)
val_encoded = encode_sentences(val_sentences, word2idx)
test_encoded = encode_sentences(test_sentences, word2idx)

In [ ]:
def encode_tags(tags, tag2idx):
    encoded = []

    for sentence in tags:
        encoded.append([
            tag2idx[tag]
            for tag in sentence
        ])

    return encoded

In [ ]:
train_tag_ids = encode_tags(train_tags, tag2idx)
val_tag_ids = encode_tags(val_tags, tag2idx)
test_tag_ids = encode_tags(test_tags, tag2idx)

In [ ]:
def filter_empty_sequences(sentences, tags):
    filtered_sentences = []
    filtered_tags = []
    for s, t in zip(sentences, tags):
        if len(s) > 0:
            filtered_sentences.append(s)
            filtered_tags.append(t)
    return filtered_sentences, filtered_tags

# Filter empty sequences from training data
train_encoded, train_tag_ids = filter_empty_sequences(train_encoded, train_tag_ids)
# Filter empty sequences from validation data
val_encoded, val_tag_ids = filter_empty_sequences(val_encoded, val_tag_ids)
# Filter empty sequences from test data
test_encoded, test_tag_ids = filter_empty_sequences(test_encoded, test_tag_ids)

print(f"After filtering: Training samples: {len(train_encoded)}, Validation samples: {len(val_encoded)}, Test samples: {len(test_encoded)}")

After filtering: Training samples: 4799, Validation samples: 600, Test samples: 601


In [ ]:
print(train_sentences[0])
print(train_encoded[0])

print(train_tags[0])
print(train_tag_ids[0])

['बे', 'बाथ्राखौ', 'मोनसे', 'गोजौ', 'थाखोनि', 'बिजिरनाय', 'एबा', 'स्टादिनिफ्राय', 'मिथिनो', 'मोननाय', 'जादों', '।']
[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
['DM_DMD', 'N_NST', 'QT_QTC', 'JJ', 'N_NST', 'V_VM', 'CC_CCD', 'RD_UNK', 'V_VM', 'V_VM', 'V_VM', 'RD_PUNC']
[1, 2, 3, 4, 2, 5, 6, 7, 5, 5, 5, 8]


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
class POSDataset(Dataset):
    def __init__(self, sentences, tags):
        self.sentences = sentences
        self.tags = tags

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        return self.sentences[idx], self.tags[idx]

In [ ]:
train_dataset = POSDataset(train_encoded, train_tag_ids)
val_dataset = POSDataset(val_encoded, val_tag_ids)
test_dataset = POSDataset(test_encoded, test_tag_ids)

In [ ]:
from torch.nn.utils.rnn import pad_sequence

PAD_WORD = word2idx["<PAD>"]
PAD_TAG = tag2idx["<PAD>"]

def collate_fn(batch):

    sentences, tags = zip(*batch)

    sentences = [torch.tensor(s, dtype=torch.long) for s in sentences]
    tags = [torch.tensor(t, dtype=torch.long) for t in tags]

    lengths = torch.tensor([len(s) for s in sentences])

    padded_sentences = pad_sequence(
        sentences,
        batch_first=True,
        padding_value=PAD_WORD
    )

    padded_tags = pad_sequence(
        tags,
        batch_first=True,
        padding_value=PAD_TAG
    )

    return padded_sentences, padded_tags, lengths

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

In [ ]:
!pip install TorchCRF

In [ ]:
import torch
import torch.nn as nn
from torchcrf import CRF
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


class BiLSTM_CRF(nn.Module):

    def __init__(self,
                 vocab_size,
                 tagset_size,
                 embedding_dim=100,
                 hidden_dim=256,
                 num_layers=2,
                 dropout=0.3,
                 pad_idx=0):

        super(BiLSTM_CRF, self).__init__()

        self.pad_idx = pad_idx

        # Embedding
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_idx
        )

        # BiLSTM
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim // 2,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout
        )

        # Dropout
        self.dropout = nn.Dropout(dropout)

        # Linear
        self.fc = nn.Linear(hidden_dim, tagset_size)

        # CRF
        self.crf = CRF(tagset_size)


    ###################################################
    #### FORWARD MUST BE INSIDE THE CLASS
    ###################################################
    def forward(self, sentences, lengths, tags=None):

        embeddings = self.embedding(sentences)

        packed = pack_padded_sequence(
            embeddings,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, _ = self.lstm(packed)

        output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True
        )

        output = self.dropout(output)

        emissions = self.fc(output)

        mask = (sentences != self.pad_idx)

        # torchcrf expects (seq_len, batch_size, num_tags)
        emissions = emissions.transpose(0, 1)
        mask = mask.transpose(0, 1)

        if tags is not None:

            tags = tags.transpose(0, 1)

            loss = -self.crf(
                emissions,
                tags,
                mask=mask,
                reduction="mean"
            )

            return loss

        prediction = self.crf.decode(
            emissions,
            mask=mask
        )

        return prediction

In [ ]:
sentences, tags, lengths = next(iter(train_loader))

print(sentences[:, 0])
print(lengths[:10])

mask = sentences != word2idx["<PAD>"]
print(mask[:, 0])

tensor([  827,  1505,   386,   561, 18387, 12355,     2, 15772,  1573,   183,
        10878,    75, 17724,  1606,     2,    33, 10880,     2,  1108,  3242,
          492,   433, 15467,   571,    33,  2162,  2484,  3453,  9356,  3529,
           78, 13879])
tensor([10,  9, 11, 17, 13,  6, 19, 26, 13, 10])
tensor([True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True])


In [ ]:
print(type(model.crf))
print(model.crf.batch_first)
print(model.crf)

<class 'torchcrf.CRF'>
False
CRF(num_tags=36)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BiLSTM_CRF(
    vocab_size=len(word2idx),
    tagset_size=len(tag2idx),
    embedding_dim=100,
    hidden_dim=256,
    num_layers=2,
    dropout=0.3,
    pad_idx=word2idx["<PAD>"]
).to(device)

print(model)

BiLSTM_CRF(
  (embedding): Embedding(18490, 100, padding_idx=0)
  (lstm): LSTM(100, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=36, bias=True)
  (crf): CRF(num_tags=36)
)


In [ ]:
print(hasattr(model, "forward"))

True


In [ ]:
sentences, tags, lengths = next(iter(train_loader))

loss = model(
    sentences.to(device),
    lengths,
    tags.to(device)
)

print(loss)

tensor(51.0517, device='cuda:0', grad_fn=<NegBackward0>)


In [ ]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.001)

NUM_EPOCHS = 20

for epoch in range(NUM_EPOCHS):

    model.train()

    total_loss = 0.0

    for sentences, tags, lengths in train_loader:

        sentences = sentences.to(device)
        tags = tags.to(device)

        optimizer.zero_grad()

        loss = model(sentences, lengths, tags)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | Loss = {avg_loss:.4f}")

Epoch 01/20 | Loss = 29.4876
Epoch 02/20 | Loss = 20.4198
Epoch 03/20 | Loss = 16.2276
Epoch 04/20 | Loss = 13.3224
Epoch 05/20 | Loss = 11.1337
Epoch 06/20 | Loss = 9.4097
Epoch 07/20 | Loss = 7.8733
Epoch 08/20 | Loss = 6.6045
Epoch 09/20 | Loss = 5.5291
Epoch 10/20 | Loss = 4.5669
Epoch 11/20 | Loss = 3.7744
Epoch 12/20 | Loss = 3.1055
Epoch 13/20 | Loss = 2.6028
Epoch 14/20 | Loss = 2.1772
Epoch 15/20 | Loss = 1.8697
Epoch 16/20 | Loss = 1.5525
Epoch 17/20 | Loss = 1.3325
Epoch 18/20 | Loss = 1.1438
Epoch 19/20 | Loss = 1.0091
Epoch 20/20 | Loss = 0.8969


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

def evaluate_model(model, data_loader, tag2idx, device):
    model.eval()
    all_predictions = []
    all_true_tags = []

    # Invert tag2idx to get idx2tag for decoding
    idx2tag = {idx: tag for tag, idx in tag2idx.items()}

    with torch.no_grad():
        for sentences, tags, lengths in data_loader:
            sentences = sentences.to(device)
            tags = tags.to(device)

            # Get predictions from the model
            predictions = model(sentences, lengths)

            # Flatten tags and predictions for metric calculation
            for i in range(len(predictions)): # Iterate over batch
                sentence_length = lengths[i].item()
                # Exclude PAD tokens and convert to list of tags
                true_tags = [idx2tag[t.item()] for t in tags[i, :sentence_length] if idx2tag[t.item()] != '<PAD>']
                pred_tags = [idx2tag[p] for p in predictions[i]]

                # Ensure lengths match after removing PAD for true_tags, and truncate predictions to match
                # CRF decode returns predictions for all tokens up to max sequence length, even if padded.
                # We need to consider only the actual length of the sentence.
                if len(pred_tags) > sentence_length:
                    pred_tags = pred_tags[:sentence_length]

                # Remove PAD from true_tags
                all_true_tags.extend(true_tags)
                all_predictions.extend(pred_tags)

    # Calculate metrics
    # Filter out '<PAD>' from true_tags and corresponding predictions
    # (This is already handled above by checking != '<PAD>')
    # We also need to map string tags to integers for classification_report
    # and ensure consistent labels between true and predicted

    # Create a unified list of all unique tags present in true and predicted data
    unique_tags = list(set(all_true_tags + all_predictions))
    if '<PAD>' in unique_tags:
        unique_tags.remove('<PAD>')

    # Convert string tags back to their original integer IDs for classification_report if necessary
    # Or, more simply, use a common set of labels and ignore '<PAD>'
    target_names = [tag for tag in idx2tag.values() if tag != '<PAD>']

    # Filter out PAD_TAG from true_tags and pred_tags lists
    filtered_true_tags = []
    filtered_predictions = []

    for true_tag_str, pred_tag_str in zip(all_true_tags, all_predictions):
        if true_tag_str != '<PAD>':
            filtered_true_tags.append(true_tag_str)
            filtered_predictions.append(pred_tag_str)

    accuracy = accuracy_score(filtered_true_tags, filtered_predictions)
    report = classification_report(filtered_true_tags, filtered_predictions, zero_division=0, digits=4)

    print(f"Accuracy: {accuracy:.4f}")
    print("Classification Report:")
    print(report)

In [ ]:
batch = next(iter(train_loader))

sentences, tags, lengths = batch

print("Sentence tensor shape:", sentences.shape)
print("Tag tensor shape:", tags.shape)
print("Lengths:", lengths[:5])

Sentence tensor shape: torch.Size([32, 50])
Tag tensor shape: torch.Size([32, 50])
Lengths: tensor([24, 38, 10, 14,  9])


In [ ]:
!pip install scikit-learn

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np
import torch

In [ ]:
def evaluate(model, dataloader, device, pad_tag_idx):

    model.eval()

    total_loss = 0

    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for sentences, tags, lengths in dataloader:

            sentences = sentences.to(device)
            tags = tags.to(device)

            # Validation loss
            loss = model(sentences, lengths, tags)
            total_loss += loss.item()

            # Predictions
            predictions = model(sentences, lengths)

            # Convert tags to CPU
            tags = tags.cpu().numpy()

            # Collect only valid tokens (ignore padding)
            for pred_seq, gold_seq, length in zip(predictions, tags, lengths):

                pred_seq = pred_seq[:length]
                gold_seq = gold_seq[:length]

                all_predictions.extend(pred_seq)
                all_labels.extend(gold_seq)

    avg_loss = total_loss / len(dataloader)

    accuracy = accuracy_score(all_labels, all_predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    return avg_loss, accuracy, precision, recall, f1

In [ ]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.001)

NUM_EPOCHS = 20

best_f1 = 0

for epoch in range(NUM_EPOCHS):

    model.train()

    train_loss = 0

    for sentences, tags, lengths in train_loader:

        sentences = sentences.to(device)
        tags = tags.to(device)

        optimizer.zero_grad()

        loss = model(sentences, lengths, tags)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    val_loss, val_acc, val_prec, val_rec, val_f1 = evaluate(
        model,
        val_loader,
        device,
        tag2idx["<PAD>"]
    )

    print("="*60)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Accuracy   : {val_acc:.4f}")
    print(f"Precision  : {val_prec:.4f}")
    print(f"Recall     : {val_rec:.4f}")
    print(f"F1-score   : {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "best_bilstm_crf.pt")
        print("✅ Best model saved!")

Epoch 1/20
Train Loss : 0.8295
Val Loss   : 32.2760
Accuracy   : 0.7304
Precision  : 0.7487
Recall     : 0.7304
F1-score   : 0.7212
✅ Best model saved!
Epoch 2/20
Train Loss : 0.7180
Val Loss   : 31.4827
Accuracy   : 0.7340
Precision  : 0.7464
Recall     : 0.7340
F1-score   : 0.7261
✅ Best model saved!
Epoch 3/20
Train Loss : 0.6413
Val Loss   : 34.4005
Accuracy   : 0.7313
Precision  : 0.7490
Recall     : 0.7313
F1-score   : 0.7214
Epoch 4/20
Train Loss : 0.5829
Val Loss   : 34.5794
Accuracy   : 0.7353
Precision  : 0.7533
Recall     : 0.7353
F1-score   : 0.7275
✅ Best model saved!
Epoch 5/20
Train Loss : 0.5238
Val Loss   : 38.9723
Accuracy   : 0.7279
Precision  : 0.7550
Recall     : 0.7279
F1-score   : 0.7166
Epoch 6/20
Train Loss : 0.4784
Val Loss   : 37.2930
Accuracy   : 0.7298
Precision  : 0.7462
Recall     : 0.7298
F1-score   : 0.7197
Epoch 7/20
Train Loss : 0.4061
Val Loss   : 37.8354
Accuracy   : 0.7315
Precision  : 0.7492
Recall     : 0.7315
F1-score   : 0.7235
Epoch 8/20
Train

In [ ]:
model.load_state_dict(torch.load("best_bilstm_crf.pt"))
model.eval()

BiLSTM_CRF(
  (embedding): Embedding(18490, 100, padding_idx=0)
  (lstm): LSTM(100, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=36, bias=True)
  (crf): CRF(num_tags=36)
)

In [ ]:
test_loss, test_acc, test_prec, test_rec, test_f1 = evaluate(
    model,
    test_loader,
    device,
    tag2idx["<PAD>"]
)

print("\nTest Results")
print("-"*40)
print(f"Loss      : {test_loss:.4f}")
print(f"Accuracy  : {test_acc:.4f}")
print(f"Precision : {test_prec:.4f}")
print(f"Recall    : {test_rec:.4f}")
print(f"F1-score  : {test_f1:.4f}")


Test Results
----------------------------------------
Loss      : 34.5240
Accuracy  : 0.7298
Precision : 0.7453
Recall    : 0.7298
F1-score  : 0.7219


In [ ]:

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def evaluate(model, dataloader, device):

    model.eval()

    total_loss = 0

    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for sentences, tags, lengths in dataloader:

            sentences = sentences.to(device)
            tags = tags.to(device)

            # Validation loss
            loss = model(sentences, lengths, tags)
            total_loss += loss.item()

            # Predictions
            predictions = model(sentences, lengths)

            tags = tags.cpu().numpy()

            # Keep only valid tokens
            for pred_seq, gold_seq, length in zip(predictions, tags, lengths):

                pred_seq = pred_seq[:length]
                gold_seq = gold_seq[:length]

                all_predictions.extend(pred_seq)
                all_labels.extend(gold_seq)

    avg_loss = total_loss / len(dataloader)

    accuracy = accuracy_score(all_labels, all_predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    return (
        avg_loss,
        accuracy,
        precision,
        recall,
        f1,
        all_labels,
        all_predictions
    )

In [ ]:
val_loss, val_acc, val_prec, val_rec, val_f1, all_labels, all_predictions = evaluate(
    model,
    val_loader,
    device
)

In [ ]:
from sklearn.metrics import classification_report

# Reverse dictionary
idx2tag = {v: k for k, v in tag2idx.items()}

# Remove PAD tag
labels = sorted(idx2tag.keys())

if tag2idx["<PAD>"] in labels:
    labels.remove(tag2idx["<PAD>"])

target_names = [idx2tag[i] for i in labels]

print(
    classification_report(
        all_labels,
        all_predictions,
        labels=labels,
        target_names=target_names,
        digits=4,
        zero_division=0
    )
)

              precision    recall  f1-score   support

      DM_DMD     0.9594    0.9220    0.9403       205
       N_NST     0.5665    0.5927    0.5793       302
      QT_QTC     0.8357    0.7238    0.7758       239
          JJ     0.6623    0.5687    0.6120       531
        V_VM     0.5831    0.9092    0.7105      1741
      CC_CCD     0.8849    0.9370    0.9102       238
      RD_UNK     0.4286    0.0833    0.1395        36
     RD_PUNC     0.9750    0.9955    0.9851      1330
        N_NN     0.7743    0.7006    0.7356      2351
       N_NNP     0.8034    0.4163    0.5484      1021
          RB     0.6702    0.4038    0.5040       156
      PR_PRP     0.8241    0.7876    0.8054       113
         PSP     0.8493    0.6889    0.7607       180
      QT_QTO     0.6349    0.5970    0.6154        67
      V_VAUX     0.6111    0.5238    0.5641        21
      RD_ECH     0.8235    0.3889    0.5283        36
      QT_QTF     0.3400    0.4474    0.3864        38
      PR_PRI     0.5676    